In [ ]:
# Cell A: Imports and setup
import pandas as pd
import numpy as np

np.random.seed(42)

In [ ]:
# Cell B: Create customers DataFrame
customers = pd.DataFrame({
    'customer_id': range(1, 101),
    'name': [f'Customer_{i}' for i in range(1, 101)],
    'region': np.random.choice(['North', 'South', 'East', 'West'], 100),
    'signup_year': np.random.choice([2020, 2021, 2022, 2023], 100)
})

In [ ]:
# Cell C: Create products DataFrame
products = pd.DataFrame({
    'product_id': range(1, 51),
    'product_name': [f'Product_{i}' for i in range(1, 51)],
    'category': np.random.choice(['Electronics', 'Clothing', 'Food', 'Books'], 50),
    'price': np.random.uniform(10, 500, 50).round(2)
})

In [ ]:
# Cell D: Create orders DataFrame (references customers and products)
n_orders = 500
orders = pd.DataFrame({
    'order_id': range(1, n_orders + 1),
    'customer_id': np.random.choice(customers['customer_id'], n_orders),
    'product_id': np.random.choice(products['product_id'], n_orders),
    'quantity': np.random.randint(1, 10, n_orders),
    'order_date': pd.date_range('2023-01-01', periods=n_orders, freq='H')
})

In [ ]:
# Cell E: Merge orders with products to get prices
orders_with_prices = orders.merge(products[['product_id', 'price', 'category']], on='product_id')
orders_with_prices['total'] = orders_with_prices['quantity'] * orders_with_prices['price']

In [ ]:
# Cell F: Merge with customers to get regions
full_orders = orders_with_prices.merge(customers[['customer_id', 'region', 'signup_year']], on='customer_id')

In [ ]:
# Cell G: Aggregate sales by region
sales_by_region = full_orders.groupby('region').agg({
    'total': ['sum', 'mean', 'count'],
    'quantity': 'sum'
}).round(2)
sales_by_region.columns = ['total_sales', 'avg_order', 'num_orders', 'total_quantity']

In [ ]:
# Cell H: Aggregate sales by category
sales_by_category = full_orders.groupby('category').agg({
    'total': ['sum', 'mean'],
    'order_id': 'count'
}).round(2)
sales_by_category.columns = ['total_sales', 'avg_order', 'num_orders']

In [ ]:
# Cell I: Top customers by total spending
customer_spending = full_orders.groupby('customer_id')['total'].sum().reset_index()
customer_spending.columns = ['customer_id', 'total_spent']
top_customers = customer_spending.nlargest(10, 'total_spent')
top_customers = top_customers.merge(customers[['customer_id', 'name', 'region']], on='customer_id')

In [ ]:
# Cell J: Summary statistics
summary = {
    'total_customers': len(customers),
    'total_products': len(products),
    'total_orders': len(orders),
    'total_revenue': full_orders['total'].sum(),
    'avg_order_value': full_orders['total'].mean(),
    'top_region': sales_by_region['total_sales'].idxmax(),
    'top_category': sales_by_category['total_sales'].idxmax()
}